# 15 history (g3)

Every archived edition of the lists, from the Internet Archive. What you should see: every edition found has an outcome; every archive prefix contributed editions; every 'no-as-of' reason still holds when the file is re-read; about 150 editions stored from 2008 to 2026, most not yet readable row by row (older layouts) but every EirGrid contracted edition since 2015 with its printed total. Then the exhibit: how the transmission queue grew.

In [ ]:
import subprocess

from iegrid import store
from iegrid.checks import (
    backfill_outcomes,
    editions,
    prefixes_without_editions,
    recheck_no_as_of,
    reupload_lag,
    same_as_of_pairs,
    unexplained_editions,
)

snap = store.latest_snapshot()
sha = subprocess.run(["git", "-C", str(store.ROOT), "rev-parse", "--short", "HEAD"],
                     capture_output=True, text=True, check=True).stdout.strip()
con = store.connect_read_only(snap)
print(f"snapshot {snap.name}  git {sha}")

## done-when: every edition found is stored or has a written reason

In [ ]:
found = backfill_outcomes(con)
for f in found:
    print(f"{f.source_id!s:36} {f.outcome:16} {f.editions:4}")
missing = unexplained_editions(con)
assert missing == 0, f"{missing} editions found with no publication and no reason"
gaps = prefixes_without_editions(con)
assert not gaps, f"archive prefixes with no editions (index down when the backfill ran?): {gaps}"
stale = recheck_no_as_of(con)
assert not stale, f"'no-as-of' reasons that no longer hold: {stale}"
print(f"\n{sum(f.editions for f in found)} editions found, every one stored or explained; "
      "every prefix searched; every no-as-of reason re-checked")

## what is stored, per list

In [ ]:
print(editions(con))

## two files, one as-of date (re-issues), and files re-uploaded long after their as-of

In [ ]:
print(same_as_of_pairs(con))
print()
print(reupload_lag(con))

## exhibit: how the queue has grown

In [ ]:
from iegrid import exhibits

fig = exhibits.queue_growth(con)
exhibits.save(fig, "g3_queue_growth")